<style>
.jp-RenderedHTMLCommon table,
.rendered_html table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
.jp-RenderedHTMLCommon th,
.jp-RenderedHTMLCommon td,
.rendered_html th,
.rendered_html td {
  text-align: left !important;
}
</style>

# 06 - Scaling, Normalization, and Encoding

<div style="background:#F7FAFC; border:1px solid #D9E2EC; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/goal.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Notebook Header</b><br>
<b>Day:</b> Day 1 - EDA and preprocessing foundations<br>
<b>Difficulty:</b> Beginner<br>
<b>Estimated time:</b> 80-100 minutes<br>
<b>Prerequisites:</b> Notebook 05 completed<br>
<b>Output:</b> You will transform numeric and categorical columns into analysis-ready formats.<br>
<b>Next notebook:</b> Day 1 mini EDA project
</div>

<details>
<summary><b>Instructor talk track</b></summary>

This notebook is about preparing columns. Keep the explanation practical: numeric columns can be on very different scales, and text categories need to be converted into numeric form for many later workflows. Do not rush the difference between standardization and normalization.

</details>

## <img src="../../../assets/icons/map.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Learning Map

By the end of this notebook, you should be able to:

- Explain why numeric columns sometimes need scaling
- Apply standard scaling
- Apply min-max normalization
- Explain the difference between `fit`, `transform`, and `fit_transform`
- Explain why categorical columns need encoding
- Apply one-hot encoding
- Apply simple ordered mapping
- Check transformed columns for row alignment and missing values
- Save a preprocessing-ready dataset

## <img src="../../../assets/icons/loop.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Signature Learning Loop

```text
QUESTION -> DATA -> CODE -> EVIDENCE -> DECISION
```


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 1: Why Preprocessing Matters

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Preprocessing means converting data into a cleaner and more useful format for analysis.
</div>

### Two common preprocessing needs

- Numeric columns may use very different scales.
- Text categories may need to become numeric columns.

Example:

- `monthly_spend` may range from hundreds to thousands.
- `visits_per_month` may range from 1 to 6.
- `membership` is text such as Gold, Silver, and Bronze.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Import libraries and load the cleaned customer dataset.
</div>


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

current = Path.cwd().resolve()
project_root = current

for candidate in [current, *current.parents]:
    if candidate.name == "applied_ds_ml":
        project_root = candidate
        break

data_path = project_root / "data" / "customer_activity_clean.csv"

if not data_path.exists():
    raise FileNotFoundError("Run Notebook 03 first so data/customer_activity_clean.csv is created.")

customers = pd.read_csv(data_path)
customers


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Dataset for Preprocessing

Before transforming columns, decide what role each column plays.

- Identifier columns help us track rows but should not be scaled or encoded as features.
- Numeric columns can be scaled or normalized.
- Categorical columns can be encoded.
- Readable text columns are useful for learning and checking, even if later models may not use them directly.

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Teaching sentence</b><br>
Preprocessing is not automatic. First decide which columns should be transformed and which columns should be kept only for reference.
</div>

In [ ]:
preprocessing_plan = pd.DataFrame({
    "column": customers.columns,
    "example_value": [customers[column].iloc[0] for column in customers.columns],
    "dtype": [customers[column].dtype for column in customers.columns],
    "preprocessing_decision": [
        "keep as row identifier",
        "keep for readability, usually not a model feature",
        "one-hot encode",
        "one-hot encode or ordered map if order is meaningful",
        "scale or normalize",
        "scale or normalize",
        "one-hot encode if used as category",
    ],
})

preprocessing_plan

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 2: Compare Column Scales

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Scale means the size range of values in a column.
</div>

### Why scale matters

If one column has large values and another column has small values, the large-valued column can dominate some calculations later.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Compare basic summaries of numeric columns.
</div>


In [ ]:
numeric_columns = ["monthly_spend", "visits_per_month"]

customers[numeric_columns].describe().round(2)


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Column Scales

- `monthly_spend` has larger values.
- `visits_per_month` has smaller values.
- Scaling helps put numeric columns into more comparable forms.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Mini Practice: Compare Ranges Directly

`describe()` gives many statistics. For scale comparison, range is the simplest first check.

```text
range = maximum - minimum
```

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Calculate ranges for the numeric columns.
</div>

In [ ]:
scale_comparison = pd.DataFrame({
    "minimum": customers[numeric_columns].min(),
    "maximum": customers[numeric_columns].max(),
    "range": customers[numeric_columns].max() - customers[numeric_columns].min(),
    "mean": customers[numeric_columns].mean(),
}).round(2)

scale_comparison

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 3: Standard Scaling

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Standard scaling changes values so the column has mean near 0 and standard deviation near 1.
</div>

### Simple meaning

Standard scaling answers:

- Is this value above or below average?
- How far is it from average in standard-deviation units?

### Formula idea

```text
standard scaled value = (value - mean) / standard deviation
```

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Apply standard scaling to numeric columns.
</div>


In [ ]:
standard_scaler = StandardScaler()
standard_scaled_values = standard_scaler.fit_transform(customers[numeric_columns])

standard_scaled_df = pd.DataFrame(
    standard_scaled_values,
    columns=["monthly_spend_standard_scaled", "visits_per_month_standard_scaled"]
)

standard_scaled_df.round(3)


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Standard Scaling

- Values near `0` are close to average.
- Positive values are above average.
- Negative values are below average.
- Larger absolute values are farther from average.


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 4: Standard Scaling by Hand

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A standard-scaled value tells how far a value is from the column average.
</div>

### Reading standard-scaled values

- `0` means close to average.
- `1` means about one standard deviation above average.
- `-1` means about one standard deviation below average.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Manually standard scale `monthly_spend` so the formula becomes visible.
</div>

In [ ]:
spend_mean = customers["monthly_spend"].mean()
spend_std_population = customers["monthly_spend"].std(ddof=0)

manual_standard_scaled_spend = (
    (customers["monthly_spend"] - spend_mean) / spend_std_population
)

manual_standard_check = pd.DataFrame({
    "monthly_spend": customers["monthly_spend"],
    "difference_from_mean": customers["monthly_spend"] - spend_mean,
    "manual_standard_scaled": manual_standard_scaled_spend,
    "sklearn_standard_scaled": standard_scaled_df["monthly_spend_standard_scaled"],
}).round(3)

manual_standard_check

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 5: Check Standard Scaled Summary

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
After standard scaling, the mean should be close to 0 and the standard deviation should be close to 1.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Summarize the standard-scaled columns.
</div>


In [ ]:
standard_scaled_df.describe().round(3)


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Standard Scaled Summary

The means should be very close to `0`.

The standard deviations may show `1.061` instead of exactly `1` in Pandas `describe()` because Pandas reports sample standard deviation by default, while `StandardScaler` uses population standard deviation.

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Teaching sentence</b><br>
Do not panic over tiny differences caused by calculation conventions. Focus on the practical result: centered near 0 and scaled to a comparable spread.
</div>

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 6: Min-Max Normalization

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Min-max normalization changes values to a fixed range, usually 0 to 1.
</div>

### Simple meaning

- `0` means the smallest value in that column.
- `1` means the largest value in that column.
- Values between 0 and 1 show relative position.

### Formula idea

```text
normalized value = (value - minimum) / (maximum - minimum)
```

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Apply min-max normalization to numeric columns.
</div>


In [ ]:
minmax_scaler = MinMaxScaler()
normalized_values = minmax_scaler.fit_transform(customers[numeric_columns])

normalized_df = pd.DataFrame(
    normalized_values,
    columns=["monthly_spend_normalized", "visits_per_month_normalized"]
)

normalized_df.round(3)


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Normalization

- The smallest value becomes `0`.
- The largest value becomes `1`.
- Other values are placed between `0` and `1`.


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 7: Min-Max Normalization by Hand

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A normalized value tells where the original value sits between the column minimum and maximum.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Manually normalize `visits_per_month` and compare with scikit-learn output.
</div>

In [ ]:
visits_min = customers["visits_per_month"].min()
visits_max = customers["visits_per_month"].max()
manual_normalized_visits = (
    (customers["visits_per_month"] - visits_min) / (visits_max - visits_min)
)

manual_normalization_check = pd.DataFrame({
    "visits_per_month": customers["visits_per_month"],
    "manual_normalized": manual_normalized_visits,
    "sklearn_normalized": normalized_df["visits_per_month_normalized"],
}).round(3)

manual_normalization_check

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Scaling vs Normalization

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Method</th>
      <th style="text-align:left; padding:6px 12px;">Result</th>
      <th style="text-align:left; padding:6px 12px;">Simple meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">Standard scaling</td><td style="text-align:left; padding:6px 12px;">Mean near 0, standard deviation near 1</td><td style="text-align:left; padding:6px 12px;">How far from average?</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Min-max normalization</td><td style="text-align:left; padding:6px 12px;">Values between 0 and 1</td><td style="text-align:left; padding:6px 12px;">Where is value between min and max?</td></tr>
      <tr><td style="text-align:left; padding:6px 12px;">Original values</td><td style="text-align:left; padding:6px 12px;">Actual business units</td><td style="text-align:left; padding:6px 12px;">What does this mean in the real world?</td></tr>
  </tbody>
</table>


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Checkpoint 1: Numeric Transformations

Answer these before continuing:

1. What does standard scaling do?
2. What does min-max normalization do?
3. Which method creates values between `0` and `1`?
4. Which method helps explain distance from average?

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Expected thinking</b><br>
Both methods change numeric values, but they answer different practical questions.
</div>


## <img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Caution: `fit`, `transform`, and `fit_transform`

Scalers learn numbers from data.

- `fit`: learn the mean, standard deviation, minimum, or maximum.
- `transform`: apply the learned transformation.
- `fit_transform`: do both steps on the same data.

For this beginner notebook, using `fit_transform` on the full learning dataset is acceptable. In real model training, fit preprocessing only on training data, then transform validation or test data with the same fitted object.

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Important</b><br>
Do not learn preprocessing settings from test data in a real modeling workflow. That creates data leakage.
</div>

In [ ]:
scaler_settings = pd.DataFrame({
    "column": numeric_columns,
    "standard_scaler_mean": standard_scaler.mean_,
    "standard_scaler_scale": standard_scaler.scale_,
    "minmax_data_min": minmax_scaler.data_min_,
    "minmax_data_max": minmax_scaler.data_max_,
}).round(3)

scaler_settings

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 8: Why Encoding Is Needed

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Encoding means converting text categories into numeric columns.
</div>

### Why encode categories?

Many later data workflows need numbers. Text values like `Gold`, `Silver`, and `Bronze` must be represented in numeric form.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Inspect categorical columns before encoding.
</div>


In [ ]:
categorical_columns = ["city", "membership", "signup_month"]

for column in categorical_columns:
    print(column)
    print(customers[column].unique())
    print()


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Categorical Values

Before encoding, check the actual category values.

- Extra spaces or inconsistent spelling create extra encoded columns.
- Missing values need a clear decision before encoding.
- New categories can appear later in real projects.

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Teaching sentence</b><br>
Clean categories before encoding, because encoding will preserve whatever category names are present.
</div>

In [ ]:
category_summary = []

for column in categorical_columns:
    category_summary.append({
        "column": column,
        "unique_count": customers[column].nunique(dropna=False),
        "has_missing": customers[column].isna().any(),
        "values": ", ".join(sorted(customers[column].dropna().astype(str).unique())),
    })

pd.DataFrame(category_summary)

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 9: One-Hot Encoding

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
One-hot encoding creates separate 0/1 columns for each category.
</div>

### Example

If `city` contains `Mumbai`, `Delhi`, and `Chennai`, one-hot encoding creates columns like:

- `city_Mumbai`
- `city_Delhi`
- `city_Chennai`

A row gets `1` for the matching city and `0` for the others.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
One-hot encode the `city` column.
</div>


In [ ]:
city_encoded = pd.get_dummies(customers["city"], prefix="city", dtype=int)

city_encoded


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: One-Hot Encoding

- Each city became a separate column.
- `1` means the row belongs to that city.
- `0` means the row does not belong to that city.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Mini Practice: Validate One-Hot Rows

For one-hot encoding of a single column, each row should usually have exactly one `1` across the encoded columns.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Check that each customer has exactly one encoded city.
</div>

In [ ]:
city_encoding_check = pd.DataFrame({
    "city": customers["city"],
    "encoded_city_sum": city_encoded.sum(axis=1),
})

city_encoding_check

## <img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Caution: `drop_first` and Lost Readability

Some workflows use `drop_first=True` to remove one dummy column and reduce duplicate information.

For Day 1 learning, keeping all one-hot columns is clearer because students can see every category directly.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Compare full one-hot encoding with `drop_first=True`.
</div>

In [ ]:
city_encoded_drop_first = pd.get_dummies(customers["city"], prefix="city", dtype=int, drop_first=True)

print("Full one-hot columns:", city_encoded.shape[1])
print("With drop_first columns:", city_encoded_drop_first.shape[1])
city_encoded_drop_first.head()

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 10: Encode Multiple Columns

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
We can one-hot encode multiple categorical columns at the same time.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
One-hot encode `city` and `membership` together.
</div>


In [ ]:
encoded_categories = pd.get_dummies(customers[["city", "membership"]], dtype=int)

encoded_categories


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 11: Ordered Mapping

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Ordered mapping converts categories to numbers when the categories have a meaningful order.
</div>

### Example order

For membership:

```text
Bronze < Silver < Gold
```

So we can map:

```text
Bronze = 1, Silver = 2, Gold = 3
```

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Caution</b><br>
Use ordered mapping only when the categories truly have an order.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Create a membership level column.
</div>


In [ ]:
membership_order = {
    "Bronze": 1,
    "Silver": 2,
    "Gold": 3,
    "Unknown": 0,
}

customers["membership_level"] = customers["membership"].map(membership_order)

customers[["membership", "membership_level"]]


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Ordered Mapping

Check for missing mapped values after mapping.

A missing mapped value can mean:

- The category was not included in the mapping dictionary.
- The category spelling is different from what the mapping expects.
- The category should not have been ordered in the first place.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Check whether every membership value received a level.
</div>

In [ ]:
mapping_check = customers[["membership", "membership_level"]].copy()

print("Missing mapped values:", mapping_check["membership_level"].isna().sum())
mapping_check

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Encoding Summary Table

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Method</th>
      <th style="text-align:left; padding:6px 12px;">Use when</th>
      <th style="text-align:left; padding:6px 12px;">Example</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">One-hot encoding</td><td style="text-align:left; padding:6px 12px;">Categories have no natural order</td><td style="text-align:left; padding:6px 12px;">City</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Ordered mapping</td><td style="text-align:left; padding:6px 12px;">Categories have a meaningful order</td><td style="text-align:left; padding:6px 12px;">Membership level</td></tr>
      <tr><td style="text-align:left; padding:6px 12px;">Keep readable columns</td><td style="text-align:left; padding:6px 12px;">Learning, checking, and reporting</td><td style="text-align:left; padding:6px 12px;">Name, city, membership</td></tr>
  </tbody>
</table>


## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Preprocessing Decision Guide

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr><th style="text-align:left; padding:6px 12px;">Column situation</th><th style="text-align:left; padding:6px 12px;">Common choice</th><th style="text-align:left; padding:6px 12px;">Reason</th></tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">Numeric values on different scales</td><td style="text-align:left; padding:6px 12px;">Standard scale or normalize</td><td style="text-align:left; padding:6px 12px;">Make numeric columns more comparable</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Categories with no natural order</td><td style="text-align:left; padding:6px 12px;">One-hot encode</td><td style="text-align:left; padding:6px 12px;">Avoid fake order</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Categories with meaningful order</td><td style="text-align:left; padding:6px 12px;">Ordered mapping</td><td style="text-align:left; padding:6px 12px;">Preserve rank information</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Names and IDs</td><td style="text-align:left; padding:6px 12px;">Keep for reference</td><td style="text-align:left; padding:6px 12px;">Useful for checking rows, not usually analysis features</td></tr>
  </tbody>
</table>

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 12: Create a Preprocessing-Ready Table

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A preprocessing-ready table combines useful original columns with transformed columns.
</div>

### What we will include

- Customer identity columns
- Original categorical columns for readability
- Scaled numeric columns
- Encoded categorical columns

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Build a preprocessing-ready DataFrame.
</div>


In [ ]:
readable_columns = customers[["customer_id", "name", "city", "membership", "signup_month"]]

preprocessed_customers = pd.concat(
    [
        readable_columns.reset_index(drop=True),
        standard_scaled_df.reset_index(drop=True),
        normalized_df.reset_index(drop=True),
        encoded_categories.reset_index(drop=True),
        customers[["membership_level"]].reset_index(drop=True),
    ],
    axis=1
)

preprocessed_customers


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Preprocessed Table

Before saving, check that the new table still has the expected number of rows.

- Row count should match the cleaned dataset.
- Transformed columns should not introduce missing values.
- Encoded columns should have clear names.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Run basic quality checks on the preprocessed table.
</div>

In [ ]:
preprocessed_quality_checks = pd.DataFrame({
    "check": [
        "cleaned row count",
        "preprocessed row count",
        "preprocessed column count",
        "missing values in preprocessed table",
    ],
    "value": [
        customers.shape[0],
        preprocessed_customers.shape[0],
        preprocessed_customers.shape[1],
        preprocessed_customers.isna().sum().sum(),
    ],
})

preprocessed_quality_checks

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 13: Save the Preprocessed Dataset

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Saving transformed data lets later notebooks reuse the same prepared table.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Save the preprocessed dataset.
</div>


In [ ]:
output_path = project_root / "data" / "customer_activity_preprocessed.csv"
preprocessed_customers.to_csv(output_path, index=False)

print("Saved preprocessed dataset to:", output_path)
print("Rows and columns:", preprocessed_customers.shape)


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Saved File

The saved CSV is the handoff to Notebook 07.

A good saved preprocessing file should be:

- Reproducible from the notebook
- Clearly named
- Stored separately from the raw and cleaned files
- Easy to inspect with Pandas

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Reload the saved file to confirm it can be used later.
</div>

In [ ]:
reloaded_preprocessed = pd.read_csv(output_path)

print("Reloaded shape:", reloaded_preprocessed.shape)
reloaded_preprocessed.head()

## <img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Common Mistakes to Avoid

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<b>Read this before practice.</b>
</div>

- Thinking standard scaling and normalization are the same.
- Forgetting that standard-scaled values can be negative.
- Expecting normalized values to be below 0 or above 1.
- Fitting scalers on test data in a real modeling workflow.
- Using ordered mapping for categories that do not have a real order.
- Forgetting to check whether every category was encoded or mapped.
- Deleting original readable columns too early.
- Forgetting to save the transformed dataset.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Practice Task

Complete these tasks:

1. Standard scale only `monthly_spend` and inspect the result.
2. Normalize only `visits_per_month` and inspect the result.
3. One-hot encode `signup_month`.
4. Explain why `city` should use one-hot encoding instead of ordered mapping.
5. Add the encoded `signup_month` columns to `preprocessed_customers`.
6. Check that the final practice table has the same number of rows as `customers`.
7. Write two plain-English explanations: one for scaling and one for encoding.

## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Exit Ticket

Answer these before closing the notebook:

1. What does standard scaling do?
2. What does min-max normalization do?
3. What is one-hot encoding?
4. When is ordered mapping acceptable?
5. What is the difference between `fit` and `transform`?
6. Why should we keep readable columns while learning?

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Ready for the Next Notebook

You are ready for the next notebook if you can:

- Compare numeric column scales.
- Apply standard scaling.
- Apply min-max normalization.
- One-hot encode categorical columns.
- Use ordered mapping only for ordered categories.
- Check transformed data before saving.
- Save a preprocessed dataset.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Practice Scaffold

Use this optional scaffold after completing the notebook. It follows the practice task order.

In [ ]:
# Practice 1: standard scale only monthly_spend
practice_standard_scaler = StandardScaler()
practice_monthly_spend_scaled = practice_standard_scaler.fit_transform(customers[["monthly_spend"]])

practice_monthly_spend_scaled_df = pd.DataFrame(
    practice_monthly_spend_scaled,
    columns=["monthly_spend_standard_scaled_practice"]
)

# Practice 2: normalize only visits_per_month
practice_minmax_scaler = MinMaxScaler()
practice_visits_normalized = practice_minmax_scaler.fit_transform(customers[["visits_per_month"]])

practice_visits_normalized_df = pd.DataFrame(
    practice_visits_normalized,
    columns=["visits_per_month_normalized_practice"]
)

# Practice 3: one-hot encode signup_month
signup_month_encoded = pd.get_dummies(customers["signup_month"], prefix="signup_month", dtype=int)

# Practice 5 and 6: combine and check row count
practice_preprocessed_customers = pd.concat(
    [
        preprocessed_customers.reset_index(drop=True),
        signup_month_encoded.reset_index(drop=True),
    ],
    axis=1,
)

print("Original rows:", customers.shape[0])
print("Practice preprocessed rows:", practice_preprocessed_customers.shape[0])
print("Practice preprocessed columns:", practice_preprocessed_customers.shape[1])

practice_preprocessed_customers.head()